# Notebook 3 — Unsupervised: Clustering dei Tratti Stradali

**Obiettivo:** segmentare la mappa di Ann Arbor in **cluster di tipologie di tratto stradale** basati sul comportamento aggregato dei veicoli che li percorrono. Il risultato è una mappa interattiva colorata della città dove ogni colore rappresenta uno *stile di guida ottimale* (urbano lento, rettilineo veloce in pianura, salita ripida, incrocio congestionato, etc.).

**Perché ha senso:** un cruise control adattivo predittivo non deve solo predire il consumo (Notebook 2): deve anche *riconoscere il tipo di tratto* per applicare la strategia giusta. Il clustering ci dà queste categorie in modo data-driven, senza etichette manuali.

**Tecniche del corso usate:**
- Standardizzazione (StandardScaler, con motivazione esplicita)
- PCA con interpretazione delle componenti
- K-Means con K-Means++ e n_init alto
- Elbow method + silhouette score per la scelta di $k$
- Caratterizzazione dei cluster con feature medie
- (Bonus) t-SNE per visualizzazione alternativa
- Visualizzazione geografica con Folium

**Input:** `outputs/ved_enriched.parquet`

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', 30)
sns.set_style('whitegrid')
RANDOM_STATE = 42

DATA = Path("./outputs/ved_enriched.parquet")
print(f"Carico {DATA}")
df = pd.read_parquet(DATA)
print(f"Shape: {df.shape}")

## 2. Aggregazione spaziale — il segmento è l'unità del clustering

**L'unità del clustering NON è la singola riga**, ma il **segmento stradale**. Definiamo un segmento come una cella della griglia geografica (~50×50 m, ottenuta arrotondando lat/lon a 4 decimali).

Per ogni cella calcoliamo statistiche aggregate del comportamento dei veicoli che ci sono passati:
- velocità media e std
- accelerazione media e std (variabilità → traffico/incroci)
- MAF medio (consumo tipico)
- RPM medio
- pendenza
- numero di passaggi (per filtrare celle con dati scarsi)

In [ ]:
# Bin spaziali da ~50m
df['lat_bin'] = df['Latitude_deg'].round(4)
df['lon_bin'] = df['Longitude_deg'].round(4)

agg = df.groupby(['lat_bin', 'lon_bin']).agg(
    n_passages=('MAF_g_per_sec', 'size'),
    speed_mean=('Vehicle_Speed_km_per_h', 'mean'),
    speed_std=('Vehicle_Speed_km_per_h', 'std'),
    accel_mean=('accel_kmh_s', 'mean'),
    accel_std=('accel_kmh_s', 'std'),
    accel_abs_mean=('accel_kmh_s', lambda s: s.abs().mean()),  # entropia di accelerazione
    maf_mean=('MAF_g_per_sec', 'mean'),
    rpm_mean=('Engine_RPM_RPM', 'mean'),
    load_mean=('Absolute_Load_pct', 'mean'),
    slope_mean=('slope', 'mean'),
    elevation=('elevation_m', 'mean'),
).reset_index()

# Frazione di stop (speed=0)
stops = df.groupby(['lat_bin', 'lon_bin'])['Vehicle_Speed_km_per_h'].apply(
    lambda s: (s < 2).mean()).reset_index(name='stop_fraction')
agg = agg.merge(stops, on=['lat_bin', 'lon_bin'])

print(f"Celle totali: {len(agg):,}")
print(f"Range n_passages: [{agg['n_passages'].min()}, {agg['n_passages'].max()}]")
agg.head()

### 2.1 Filtro: scarta celle con dati scarsi

Celle con pochi passaggi sono rumorose: una statistica fatta su 3 punti non è significativa. Imponiamo una soglia minima (es. 50 osservazioni).

In [ ]:
MIN_PASSAGES = 50
before = len(agg)
agg = agg[agg['n_passages'] >= MIN_PASSAGES].reset_index(drop=True)
print(f"Celle dopo filtro (n_passages >= {MIN_PASSAGES}): {len(agg):,} (rimosse {before-len(agg):,})")

# Visualizza la mappa dei dati rimanenti
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(agg['lon_bin'], agg['lat_bin'], c=agg['n_passages'],
                cmap='viridis', s=10, alpha=0.8)
plt.colorbar(sc, ax=ax, label='N. passaggi per cella')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Celle filtrate (≥ {MIN_PASSAGES} passaggi)')
plt.show()

## 3. Selezione delle feature per il clustering

Feature scelte: comportamento aggregato (velocità, accelerazione, MAF, RPM, load, fermate, pendenza). Escludiamo `n_passages` (è un meta-dato), `elevation` (assoluta, non utile per *tipo* di tratto), e le coordinate (non vogliamo che i cluster siano "vicinato A vs vicinato B", ma *tipologia* di guida).

In [ ]:
FEATURES_CLUSTER = [
    'speed_mean', 'speed_std',
    'accel_mean', 'accel_std', 'accel_abs_mean',
    'maf_mean', 'rpm_mean', 'load_mean',
    'slope_mean', 'stop_fraction'
]

X = agg[FEATURES_CLUSTER].copy()
X = X.dropna()  # alcune celle potrebbero avere std NaN con un solo passaggio
agg_clean = agg.loc[X.index].reset_index(drop=True)
X = X.reset_index(drop=True)
print(f"Celle finali per clustering: {len(X):,}")
print(f"Feature: {len(FEATURES_CLUSTER)}")

X.describe()

## 4. Standardizzazione — perché è obbligatoria

Guarda le scale delle feature:
- `speed_mean`: 0–100 km/h
- `slope_mean`: -0.1 a 0.1
- `accel_std`: 0–5
- `rpm_mean`: 0–3000

Senza standardizzazione, K-Means calcolerebbe la distanza euclidea tra punti dominata da `rpm_mean` (range più ampio), e `slope_mean` sarebbe trascurata. Lo standardscaler riporta tutte le feature a media 0 e std 1.

In [ ]:
# Esempio: cluster fatti SENZA standardizzazione (didattico)
demo_km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(X)
print("Senza scaling, importanza relativa delle feature nella distanza euclidea:")
var_contrib = X.var() / X.var().sum()
print((var_contrib * 100).round(2).sort_values(ascending=False))
print("\n→ Le feature con range grande dominano TUTTO il clustering. Inaccettabile.\n")

# Ora con scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_CLUSTER)
print("Dopo standardizzazione:")
print(X_scaled_df.describe().loc[['mean', 'std']].round(3))

## 5. Scelta di K — Elbow Method + Silhouette

Proviamo K da 2 a 10. Per ogni K calcoliamo:
- **inerzia** (WCSS): somma dei quadrati delle distanze dai centroidi. Decresce sempre con K, ma il punto in cui rallenta è il "gomito".
- **silhouette score**: misura quanto i cluster sono ben separati. Va da -1 a 1, più alto è meglio.

In [ ]:
K_VALUES = list(range(2, 11))
inertias = []
silhouettes = []

# Silhouette è costosa: usiamo un sample per K alti
if len(X_scaled) > 10000:
    sil_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), size=10000, replace=False)
else:
    sil_idx = np.arange(len(X_scaled))

for k in K_VALUES:
    km = KMeans(n_clusters=k, n_init=20, init='k-means++', random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled[sil_idx], labels[sil_idx])
    silhouettes.append(sil)
    print(f"  k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(K_VALUES, inertias, 'o-', color='steelblue', lw=2, markersize=10)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_VALUES, silhouettes, 'o-', color='coral', lw=2, markersize=10)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette Method')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.1 Scelta finale di K

Combinando elbow (dove la pendenza dell'inerzia cambia bruscamente) e silhouette (massimo locale), tipicamente troveremo un buon valore tra **4 e 6**. Scegli il K che vedi essere il gomito; modifica `K_FINAL` se diverso.

In [ ]:
# Default: prendiamo il K con miglior silhouette tra i "ragionevoli" (≥3, ≤7)
K_FINAL = K_VALUES[np.argmax([s if 3 <= k <= 7 else -1 for k, s in zip(K_VALUES, silhouettes)])]
print(f"K scelto: {K_FINAL}")
print("Puoi modificare K_FINAL manualmente se preferisci un altro valore basandoti sul gomito.")

## 6. K-Means finale

Eseguiamo il K-Means con K scelto, `n_init=50` per ridurre il rischio di minimo locale, e K-Means++ come strategia di inizializzazione (sceglie centroidi iniziali distanti tra loro).

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, n_init=50, init='k-means++', random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)
agg_clean['cluster'] = labels

print("Distribuzione dei cluster:")
print(agg_clean['cluster'].value_counts().sort_index())

## 7. Caratterizzazione e naming dei cluster

**Questa è la sezione più importante del progetto.** Cluster senza interpretazione sono inutili. Per ogni cluster guardiamo le feature medie e gli diamo un nome interpretativo.

In [ ]:
cluster_profile = agg_clean.groupby('cluster')[FEATURES_CLUSTER + ['n_passages']].mean().round(2)
cluster_profile['count'] = agg_clean['cluster'].value_counts().sort_index()
cluster_profile

In [ ]:
# Heatmap dei profili cluster (normalizzati su tutti i cluster per leggibilità)
profile_norm = (cluster_profile[FEATURES_CLUSTER] - cluster_profile[FEATURES_CLUSTER].mean()) / cluster_profile[FEATURES_CLUSTER].std()

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(profile_norm.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Z-score (vs altri cluster)'}, ax=ax)
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')
ax.set_title(f'Profilo dei {K_FINAL} cluster (Z-score relativo)')
plt.tight_layout()
plt.show()

### 7.1 Naming dei cluster

**ATTENZIONE: i cluster K-Means non hanno ordinamento fisso.** Ogni esecuzione produce cluster con gli stessi pattern ma numerati diversamente. Devi guardare la heatmap sopra e dare un nome a ciascun cluster manualmente.

**Pattern tipici che ci aspettiamo di trovare:**
- **Urbano lento / incrocio**: speed_mean basso, stop_fraction alto, accel_std alto, accel_abs_mean alto
- **Rettilineo veloce**: speed_mean alto, speed_std basso, accel_abs_mean basso, MAF medio
- **Salita sostenuta**: slope_mean positivo significativo, load_mean alto, MAF alto a parità di velocità
- **Discesa**: slope_mean negativo, load_mean basso
- **Tratto cittadino normale**: tutto vicino alla media

Esempio di naming (da adattare guardando la TUA heatmap):

In [ ]:
# Naming automatico euristico basato sui profili (da rivedere manualmente!)
def auto_name(row):
    name = []
    if row['stop_fraction'] > cluster_profile['stop_fraction'].quantile(0.7):
        name.append('Urbano/Incrocio')
    elif row['speed_mean'] > cluster_profile['speed_mean'].quantile(0.7):
        name.append('Rettilineo veloce')
    elif row['slope_mean'] > 0.01:
        name.append('Tratto in salita')
    elif row['slope_mean'] < -0.01:
        name.append('Tratto in discesa')
    else:
        name.append('Cittadino misto')
    return ' | '.join(name) if name else 'Generico'

cluster_profile['name_auto'] = cluster_profile.apply(auto_name, axis=1)
cluster_names = cluster_profile['name_auto'].to_dict()
print("Naming proposto (verifica e modifica se necessario):")
for k, v in cluster_names.items():
    print(f"  Cluster {k}: {v}")

# Mappa nei dati
agg_clean['cluster_name'] = agg_clean['cluster'].map(cluster_names)

## 8. PCA per visualizzazione e interpretazione

Riduciamo a 2 componenti per visualizzare i cluster nel piano. La PCA è anche utile per **interpretare** cosa lo spazio delle feature "premia": guardando i loadings vediamo quali feature contribuiscono di più alle prime componenti.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"Varianza spiegata: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Totale: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Interpretazione: quali feature contribuiscono a ciascuna componente
loadings = pd.DataFrame(pca.components_.T, index=FEATURES_CLUSTER, columns=['PC1', 'PC2'])
print("\nLoadings:")
print(loadings.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Loadings PCA')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('tab10', K_FINAL)
for c in sorted(agg_clean['cluster'].unique()):
    mask = agg_clean['cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               s=12, alpha=0.6, color=palette[c],
               label=f'C{c}: {cluster_names[c]}')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Cluster nello spazio PCA (2D)')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## 9. Visualizzazione geografica — la mappa di Ann Arbor

**Questa è la sezione "deliverable visivo"** del progetto. Coloriamo la mappa di Ann Arbor con i cluster, ottenendo una vera classificazione geografica dei tipi di tratto stradale.

In [ ]:
# Plot statico (matplotlib) — sempre disponibile
fig, ax = plt.subplots(figsize=(12, 10))
for c in sorted(agg_clean['cluster'].unique()):
    mask = agg_clean['cluster'] == c
    ax.scatter(agg_clean.loc[mask, 'lon_bin'], agg_clean.loc[mask, 'lat_bin'],
               s=8, alpha=0.7, color=palette[c],
               label=f'C{c}: {cluster_names[c]}')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Cluster di tratti stradali — Ann Arbor (K={K_FINAL})')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('./outputs/cluster_map_static.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Mappa interattiva con folium (richiede: pip install folium)
try:
    import folium
    center = [agg_clean['lat_bin'].mean(), agg_clean['lon_bin'].mean()]
    m = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')
    
    # Palette hex per folium
    palette_hex = [f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'
                   for r, g, b in palette]
    
    # Subsample se troppi punti per Folium (massimo ~10k è ragionevole)
    plot_df = agg_clean.sample(n=min(10000, len(agg_clean)), random_state=RANDOM_STATE)
    
    for _, row in plot_df.iterrows():
        c = int(row['cluster'])
        folium.CircleMarker(
            location=[row['lat_bin'], row['lon_bin']],
            radius=3,
            color=palette_hex[c],
            fill=True,
            fill_opacity=0.7,
            popup=f"Cluster {c}: {cluster_names[c]}<br>"
                  f"Speed={row['speed_mean']:.1f} km/h<br>"
                  f"MAF={row['maf_mean']:.1f} g/s<br>"
                  f"Stop frac={row['stop_fraction']:.2f}"
        ).add_to(m)
    
    # Legenda HTML
    legend_html = '<div style="position: fixed; bottom: 30px; left: 30px; width: 250px; background: white; border:2px solid grey; padding: 10px; z-index: 1000;"><b>Cluster legend</b><br>'
    for c, name in cluster_names.items():
        legend_html += f'<i style="background:{palette_hex[c]}; width:12px; height:12px; display:inline-block;"></i> C{c}: {name}<br>'
    legend_html += '</div>'
    m.get_root().html.add_child(folium.Element(legend_html))
    
    OUT_MAP = Path('./outputs/cluster_map.html')
    m.save(str(OUT_MAP))
    print(f"✓ Mappa interattiva salvata: {OUT_MAP}")
    print("  Apri il file HTML nel browser per esplorarla.")
except ImportError:
    print("folium non installato. Installa con: pip install folium")
    print("Per ora va bene la mappa statica salvata sopra.")

## 10. (Bonus) t-SNE per visualizzazione non lineare

PCA è lineare e preserva struttura globale. t-SNE è non lineare e preserva struttura locale: spesso mostra cluster più "separati" visivamente.

**Limiti di t-SNE da ricordare:**
- Stocastico (risultati diversi a ogni run con random_state diverso)
- Lento su dataset grandi (campioniamo)
- Le distanze tra cluster nel plot finale **non sono interpretabili** in modo quantitativo

In [ ]:
from sklearn.manifold import TSNE

n_tsne = min(5000, len(X_scaled))
idx_tsne = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), size=n_tsne, replace=False)

print(f"t-SNE su {n_tsne} punti...")
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000,
            random_state=RANDOM_STATE, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_scaled[idx_tsne])
labels_tsne = agg_clean['cluster'].iloc[idx_tsne].values

fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(np.unique(labels_tsne)):
    mask = labels_tsne == c
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=10, alpha=0.6,
               color=palette[c], label=f'C{c}: {cluster_names[c]}')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('Cluster nello spazio t-SNE (2D)')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## 11. Confronto cluster ↔ EngineType (sanity check)

Per ogni cluster di tratto stradale, qual è la composizione di EngineType dei veicoli che lo percorrono? In teoria dovrebbe essere abbastanza uniforme (i veicoli HEV percorrono le stesse strade degli ICE), ma se c'è una differenza è interessante.

In [ ]:
# Per ogni cella prendiamo la proporzione di passaggi per EngineType
engine_breakdown = df.merge(
    agg_clean[['lat_bin', 'lon_bin', 'cluster', 'cluster_name']],
    on=['lat_bin', 'lon_bin'], how='inner'
).groupby(['cluster', 'cluster_name', 'EngineType']).size().unstack(fill_value=0)
engine_breakdown_pct = engine_breakdown.div(engine_breakdown.sum(axis=1), axis=0) * 100

print("Composizione EngineType per cluster (%):")
print(engine_breakdown_pct.round(1))

engine_breakdown_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
                          color=['steelblue', 'coral', 'gold'])
plt.ylabel('% passaggi')
plt.title('Composizione EngineType per cluster')
plt.xticks(rotation=15, ha='right')
plt.legend(title='EngineType', loc='upper right')
plt.tight_layout()
plt.show()

## 12. Salvataggio dei risultati

In [ ]:
agg_clean.to_parquet('./outputs/road_segment_clusters.parquet', index=False)
cluster_profile.to_csv('./outputs/cluster_profile.csv')
print("✓ Salvati:")
print("  outputs/road_segment_clusters.parquet (segmenti con etichetta cluster)")
print("  outputs/cluster_profile.csv (profilo numerico dei cluster)")
print("  outputs/cluster_map_static.png (mappa)")
print("  outputs/cluster_map.html (mappa interattiva se folium installato)")

---

## Riepilogo finale

**Cosa abbiamo costruito:**
- Aggregazione spaziale dei 17M punti in ~migliaia di celle (~50×50 m)
- Filtro qualità (≥ 50 passaggi per cella)
- Standardizzazione con motivazione esplicita (dimostrazione del fallimento senza scaling)
- Elbow + Silhouette per scelta di K
- K-Means con K-Means++ e n_init=50
- Caratterizzazione e naming dei cluster con heatmap z-score
- PCA con interpretazione dei loadings
- t-SNE come visualizzazione alternativa
- Mappa geografica statica + interattiva di Ann Arbor con cluster
- Confronto cluster ↔ EngineType

**Collegamento con il Notebook 2:** l'etichetta `cluster_name` potrebbe essere usata come feature aggiuntiva nel modello supervised (concettualmente: "il modello sa che il prossimo tratto sarà di tipo X"). Questo chiude il cerchio della narrativa del cruise control adattivo predittivo.

**Limitazioni (da menzionare nella presentazione):**
- K-Means assume cluster sferici e di dimensione simile; alternative come DBSCAN o HDBSCAN potrebbero trovare cluster di forma diversa
- L'aggregazione spaziale a celle quadrate è approssimativa: una segmentazione vera su grafo stradale (OpenStreetMap) sarebbe più accurata
- I cluster sono dipendenti dall'area: applicare a un'altra città richiederebbe ri-training